# NSBI training

In this notebook, you will be training surrogate neural networks that estimate the ratio of probability densities of an event under different hypotheses.

$$
\hat r(x;\mu_1,\mu_2) \approx \frac{\int dz \, p(x,z;\mu_1)}{\int dz \, p(x,z;\mu_0)}
$$

This surrogate will use the CARL method outlined in [1907.10621](https://arxiv.org/abs/1907.10621) and tested in [1808.00973](https://arxiv.org/abs/1808.00973). 

We will be estimating the density ratio of two numerator hypotheses with respect to a common denominator:

  1. the signal-only
     $$
         r( x ; {\mu_{\rm S}}, \mu_{\rm B})
     $$
  1. signal+background+interference processes over the background-only process.
     $$
        r( x ; \mu_{\rm SBI}, \mu_{\rm B})
     $$

Refer to the the data exploration notebook to recall how these estimates allow us to obtain an estimate of the full SBI process under modifications to the Higgs signal strength.This will not be realized until the completion of the final notebook. In this notebook, we limit our scope to setting up the training. 

In [ ]:
import json

import pandas as pd
import numpy as np

import torch
from torch.utils.data import TensorDataset, DataLoader
torch.set_default_dtype(torch.float32)
torch.set_float32_matmul_precision('medium')
import lightning as L

from physics.simulation import mcfm
from physics.analysis import zz4l, zz2l2v
from physics.hstar import sigstr
from nsbi import carl

import matplotlib, matplotlib.pyplot as plt

## 1. Preparing the training datasets

The training data consists of examples drawn from the two hypotheses of which we wish to estimate the ratio of. 

In [ ]:
data_dir = '/global/cfs/cdirs/ntrain6/NSBIData/carl_models/'

(events_sig_train, events_sig_val), (events_bkg_train, events_bkg_val) = carl.utils.load_data(data_dir, 'sig_over_bkg')
(events_sbi_train, events_sbi_val), _ = carl.utils.load_data(data_dir, 'sbi_over_bkg')

### 1.(a) Scale the features

It is often common to scale observables such that $\left< x \right> = 0$ and standard deviation $\sigma_x = 1$, referred to as standard scaling. Perform the following:

1. Scale the observables of the *training* data such that the above holds exactly.
2. Scale the observables of *validation* data exactly according to the scaling performed to the training data.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

features_4l = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy', 
               'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy', 
               'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy', 
               'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']

X_sig_train = scaler.fit_transform(events_sig_train.kinematics[features_4l].to_numpy())
X_sig_val = scaler.transform(events_sig_val.kinematics[features_4l].to_numpy())

X_sbi_train = scaler.fit_transform(events_sbi_train.kinematics[features_4l].to_numpy())
X_sbi_val = scaler.transform(events_sbi_val.kinematics[features_4l].to_numpy())

X_bkg_train = scaler.fit_transform(events_bkg_train.kinematics[features_4l].to_numpy())
X_bkg_val = scaler.transform(events_bkg_val.kinematics[features_4l].to_numpy())

---
**Questions for participants**
1. How would one go about scaling a set of features? 
1. Why do you think observable scaling a common practice in machine learning?
1. What could happen if the observables are not scaled?
1. Can you think of a case where you would not want to scale the observables?
---

### 1.(b) "Balance" the hypotheses

The loss funciton used in this training requires the dataset be balanced (i.e. their total rate of occurences in the training data are the same) for the training to converge. Explicitly, we require

$$
    \sum_{i=1}^N w(x_i,z_i;\mu_0) = \sum_{i=1}^N w(x_i,z_i;\mu_1)
$$

One way to enfoce this is by converting the event weights into a proability by 

$$
    p(x,z;\mu)=\frac{w(x_i,z_i;\mu)}{\sum_{i=1}^N w(x_i,z_i;\mu)}
$$

Let's enforce these for each of the datasets:

In [ ]:
w_sig_train, w_sig_val = events_sig_train.weights / events_sig_train.weights.sum(), events_sig_val.weights / events_sig_val.weights.sum()
w_sbi_train, w_sbi_val = events_sbi_train.weights / events_sbi_train.weights.sum(), events_sbi_val.weights / events_sbi_val.weights.sum()
w_bkg_train, w_bkg_val = events_bkg_train.weights / events_bkg_train.weights.sum(), events_bkg_val.weights / events_bkg_val.weights.sum()

---
**Question for participants**
1. What issues could you imagine running into if your distribution of data is not carefully accounted for?
---

## 2. Building the NN architecture

Implement the function to specify the layers of a multi-layer perceptron (MLP) with:

1. As many input nodes as there are features,
2. As many hidden layer-times-nodes as desired, all with a ReLU activation function.
4. One output node with a sigmoid activation function.

In [ ]:
import torch
from torch import nn

def nn_layers(n_features, n_layers, n_nodes):
    layers = []
    layers.append(nn.Sequential(nn.Linear(n_features, n_nodes), nn.ReLU()))
    for _ in range(n_layers):
        layers.append(nn.Sequential(nn.Linear(n_nodes, n_nodes), nn.ReLU()))
    layers.append(nn.Sequential(nn.Linear(n_nodes, 1), nn.Sigmoid()))
    return layers

---
**Questions for participants**
1. What is the importance of `nn.Sigmoid()`?
1. What changes could you make to the above network during the architecture optimizaiton process?
---

## 3. Preparing to train the DNNs

An example `torch/lightning` implementation of the above is included in this tutorial code base, and can be run using the command

```sh
 python -m nsbi.carl fit \
    --data.features '["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "l3_pt", "l3_eta", "l3_phi", "l3_energy", "l4_pt", "l4_eta", "l4_phi", "l4_energy"]' \
    --data.numerator_events '/global/cfs/cdirs/ntrain6/NSBIData/h4l_data/sm/ggzz4l_sig.csv' \
    --data.denominator_events '/global/cfs/cdirs/ntrain6/NSBIData/h4l_data/sm/ggzz4l_bkg.csv' \
    --data.denominator_reweight '["sig","bkg"]' \
    --data.batch_size BATCH_SIZE \
    --model.learning_rate LEARNING_RATE \
    --model.n_layers N_LAYERS \
    --model.n_nodes N_NODES \
    --trainer.max_epochs 500 \
    --seed_everything SEED
```

where arguments in all capital letters are defined by the user. 

---
**Questions for participants**

1. Why is it important to check the loss for epoch 0? 
1. What parameters would you expect to have to optimize in the above command?
1. Why is it critical to specify the seed?
---